In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
A = np.random.normal(50, 12, (1000, 5))

print(A.shape)
print(A.dtype)
print(A.nbytes)

A = A.astype(np.float32)
print(A.dtype)
print(A.nbytes)

(1000, 5)
float64
40000
float32
20000


In [2]:
A = (A - A.mean(axis=0)) / A.std(axis=0)

print(np.round(A.mean(axis=0), 6))
print(np.round(A.std(axis=0), 6))

[-4.e-06  2.e-06  2.e-06 -3.e-06  0.e+00]
[0.999999 1.       1.       1.       1.      ]


In [3]:
def minmax(X, axis):
    mn = X.min(axis=axis, keepdims=True)
    mx = X.max(axis=axis, keepdims=True)
    return (X - mn) / np.where(mx == mn, 1, mx - mn)

test = np.random.rand(4, 3)

scaled_0 = minmax(test, 0)
print(scaled_0.min(axis=0))
print(scaled_0.max(axis=0))

scaled_1 = minmax(test, 1)
print(scaled_1.min(axis=1))
print(scaled_1.max(axis=1))

[0. 0. 0.]
[1. 1. 1.]
[0. 0. 0. 0.]
[1. 1. 1. 1.]


In [4]:
mask = np.random.rand(*A.shape) < 0.05
A[mask] = np.nan

means = np.nanmean(A, axis=0)
r, c = np.where(np.isnan(A))
A[r, c] = means[c]

print(np.isnan(A).sum() == 0)

True


In [5]:
mean = A.mean(axis=0)
std = A.std(axis=0)
median = np.median(A, axis=0)

outliers = np.abs(A - mean) > 3 * std
print(outliers.sum(axis=0))

A = np.where(outliers, median, A)

[5 2 5 3 5]


In [6]:
X = A[:200]
diff = X[:, None, :] - X[None, :, :]
D = np.sqrt(np.sum(diff ** 2, axis=2))

print(D.shape)
print(np.allclose(np.diag(D), 0))
print(np.allclose(D, D.T))

(200, 200)
True
True


In [7]:
np.fill_diagonal(D, np.inf)
nearest = np.argpartition(D, 5, axis=1)[:, :5]

print(nearest.shape)
print(nearest)

(200, 5)
[[180 185 192 182 179]
 [199 114  17  70  31]
 [  3 158  37  99 128]
 [132  99   2 145 194]
 [160 158 102  43  99]
 [ 43 194 132 106 160]
 [ 31  38 199   1  17]
 [195 149  76 166 174]
 [ 15  37  89 157 158]
 [ 48 187  39   3  73]
 [137  23  11  58  81]
 [ 10 137  81 175 184]
 [ 66 171 128  52  38]
 [159 143 192 191 193]
 [ 40  64  15 157 161]
 [  8 157  89  39  37]
 [ 93  33 145 169 194]
 [183   1  38  95 199]
 [ 90  63  61  55 168]
 [131  41 183  73  63]
 [ 52 166  80 149 126]
 [ 70  72 114  61  42]
 [116 182   0 185  49]
 [ 10  58 139 137 190]
 [146 141 194 184 133]
 [ 92 176  44 160 136]
 [104  71 101 126  91]
 [107 117  67 136  39]
 [ 53 168 138  90 167]
 [ 36 144  82  96 117]
 [142 168  63 196  55]
 [199   1   6  98 112]
 [ 17 163   1 188 153]
 [169  16 164  93  70]
 [155  60 125 111 189]
 [ 74 122  56 178 153]
 [ 29 117 142 144 107]
 [158   8 145 102 140]
 [ 17 183   6 128  52]
 [ 67 136  27  99 160]
 [ 42 179  48 157  13]
 [123  19 131  63 192]
 [ 70  41  21 123  40]
 [

In [8]:
%%time
values = np.random.randint(0, 100, 1_000_000)
counts_numpy = np.bincount(values, minlength=100)

CPU times: user 19.2 ms, sys: 2.92 ms, total: 22.1 ms
Wall time: 22.7 ms


In [9]:
%%time
counts_python = [0] * 100
for val in values:
    counts_python[val] += 1

assert np.array_equal(counts_numpy, counts_python)

CPU times: user 210 ms, sys: 206 µs, total: 210 ms
Wall time: 211 ms


In [10]:
from numpy.lib.stride_tricks import sliding_window_view

arr = np.random.rand(10000)
windows = sliding_window_view(arr, 10)
print(windows.shape)

w_mean = windows.mean(axis=1)
r_mean = pd.Series(arr).rolling(10).mean().to_numpy()

print(np.allclose(w_mean, r_mean[9:]))

(9991, 10)
True


In [11]:
arr = np.arange(96).reshape(12, 8)
reshaped = arr.reshape(4, 3, 8)
res = reshaped.mean(axis=1)

print(reshaped.shape)
print(res.shape)
print(res)

(4, 3, 8)
(4, 8)
[[ 8.  9. 10. 11. 12. 13. 14. 15.]
 [32. 33. 34. 35. 36. 37. 38. 39.]
 [56. 57. 58. 59. 60. 61. 62. 63.]
 [80. 81. 82. 83. 84. 85. 86. 87.]]


In [12]:
np.save("clean.npy", A)
loaded = np.load("clean.npy")

assert np.array_equal(loaded, A)
print(True)

True
